# 03 - Train YOLOv8 (Baseline)

This notebook trains and evaluates two YOLOv8 variants on the license plate detection dataset:
- **YOLOv8n** (nano) — fastest, smallest
- **YOLOv8s** (small) — better accuracy, still fast

| Step | Description |
|---|---|
| 1 | Install & import `ultralytics` |
| 2 | Detect device (MPS / CUDA / CPU) |
| 3 | Train YOLOv8n for 100 epochs |
| 4 | Validate — print mAP@50, mAP@50:95, precision, recall |
| 5 | Visualise predictions on test images |
| 6 | Train YOLOv8s and compare |
| 7 | Copy best weights to `models/` |

## 1. Install & Import

In [ ]:
# On Google Colab run this cell; locally skip or install once via pip
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %pip install -q ultralytics

In [ ]:
import os
import glob
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
from ultralytics import YOLO

print(f'PyTorch  : {torch.__version__}')
try:
    import ultralytics
    print(f'Ultralytics: {ultralytics.__version__}')
except Exception:
    pass

## 2. Device Detection

In [ ]:
def get_device():
    """Return the best available device string for ultralytics."""
    if torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return '0'   # first CUDA GPU
    return 'cpu'

DEVICE = get_device()
print(f'Using device: {DEVICE}')

if DEVICE == 'mps':
    print('  Apple Silicon MPS detected — GPU acceleration enabled.')
elif DEVICE == '0':
    print(f'  CUDA GPU: {torch.cuda.get_device_name(0)}')
else:
    print('  No GPU found — training on CPU (will be slow).')

## 3. Configuration & Paths

In [ ]:
# All paths are relative to this notebook's location (notebooks/)
NOTEBOOK_DIR = Path(os.getcwd())               # .../license-plate-detection/notebooks
PROJECT_DIR  = NOTEBOOK_DIR.parent             # .../license-plate-detection

if IN_COLAB:
    # Adjust to your Drive path
    PROJECT_DIR = Path('/content/drive/MyDrive/license-plate-detection')

DATA_YAML   = PROJECT_DIR / 'data' / 'processed' / 'yolo' / 'dataset.yaml'
TEST_IMG_DIR = PROJECT_DIR / 'data' / 'processed' / 'yolo' / 'images' / 'test'
RESULTS_DIR = PROJECT_DIR / 'results' / 'yolo'
MODELS_DIR  = PROJECT_DIR / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
EPOCHS    = 100
BATCH     = 16
IMG_SIZE  = 640
PATIENCE  = 20    # early stopping patience

print('Paths')
print(f'  dataset yaml : {DATA_YAML}')
print(f'  test images  : {TEST_IMG_DIR}')
print(f'  results      : {RESULTS_DIR}')
print(f'  models       : {MODELS_DIR}')
print()
print(f'Hyperparameters: epochs={EPOCHS}, batch={BATCH}, imgsz={IMG_SIZE}, patience={PATIENCE}')

assert DATA_YAML.exists(), f'dataset.yaml not found: {DATA_YAML}\nRun 02_data_preprocessing.ipynb first.'

## 4. Train YOLOv8n (Nano)

YOLOv8n is the **fastest and smallest** variant — good baseline.  
Pretrained on COCO; we fine-tune on our license plate data.

In [ ]:
# Load pretrained YOLOv8n
model_nano = YOLO('yolov8n.pt')
print(model_nano.info())

In [ ]:
results_nano = model_nano.train(
    data      = str(DATA_YAML),
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMG_SIZE,
    device    = DEVICE,
    patience  = PATIENCE,       # stop early if no improvement
    project   = str(RESULTS_DIR),
    name      = 'yolov8n',
    exist_ok  = True,
    # Augmentation (ultralytics built-in)
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.0,
    # Logging
    plots     = True,
    save      = True,
    verbose   = True,
)

print('\nYOLOv8n training complete.')
print(f'Results saved to: {RESULTS_DIR / "yolov8n"}')

## 5. Validate YOLOv8n — Metrics

In [ ]:
# Load best checkpoint and validate on val split
best_nano = RESULTS_DIR / 'yolov8n' / 'weights' / 'best.pt'
model_nano_best = YOLO(str(best_nano))

val_nano = model_nano_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'val',
)

print('\n' + '=' * 50)
print('YOLOv8n — Validation Metrics')
print('=' * 50)
print(f'  mAP@50      : {val_nano.box.map50:.4f}')
print(f'  mAP@50:95   : {val_nano.box.map:.4f}')
print(f'  Precision   : {val_nano.box.mp:.4f}')
print(f'  Recall      : {val_nano.box.mr:.4f}')

In [ ]:
# Also evaluate on the held-out TEST split
test_nano = model_nano_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'test',
)

print('\n' + '=' * 50)
print('YOLOv8n — Test Split Metrics')
print('=' * 50)
print(f'  mAP@50      : {test_nano.box.map50:.4f}')
print(f'  mAP@50:95   : {test_nano.box.map:.4f}')
print(f'  Precision   : {test_nano.box.mp:.4f}')
print(f'  Recall      : {test_nano.box.mr:.4f}')

### 5.1 Training Curves

In [ ]:
def plot_training_curves(run_dir, model_name):
    """Plot loss and mAP curves from ultralytics results.csv."""
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        print(f'results.csv not found at {csv_path}')
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # ultralytics adds spaces

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=15)

    def safe_plot(ax, col, title, color='steelblue'):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=1.5)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'Column not found:\n{col}',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)

    safe_plot(axes[0, 0], 'train/box_loss',  'Train Box Loss',  'tomato')
    safe_plot(axes[0, 1], 'train/cls_loss',  'Train Class Loss','darkorange')
    safe_plot(axes[0, 2], 'train/dfl_loss',  'Train DFL Loss',  'goldenrod')
    safe_plot(axes[1, 0], 'metrics/mAP50(B)',    'Val mAP@50',      'steelblue')
    safe_plot(axes[1, 1], 'metrics/mAP50-95(B)', 'Val mAP@50:95',   'mediumpurple')
    safe_plot(axes[1, 2], 'val/box_loss',    'Val Box Loss',    'seagreen')

    plt.tight_layout()
    save_path = RESULTS_DIR / f'{model_name}_training_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


plot_training_curves(RESULTS_DIR / 'yolov8n', 'yolov8n')

### 5.2 Built-in Ultralytics Plots

Ultralytics automatically saves these plots to the run directory:

In [ ]:
def show_run_plots(run_dir, model_name, plot_names=None):
    """Display PNG plots saved by ultralytics in the run directory."""
    if plot_names is None:
        plot_names = ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'results.png']

    found = []
    for name in plot_names:
        p = Path(run_dir) / name
        if p.exists():
            found.append(p)

    if not found:
        print('No plots found yet — run training first.')
        return

    fig, axes = plt.subplots(1, len(found), figsize=(7 * len(found), 6))
    if len(found) == 1:
        axes = [axes]

    for ax, p in zip(axes, found):
        img = np.array(Image.open(p))
        ax.imshow(img)
        ax.set_title(p.name, fontsize=11)
        ax.axis('off')

    fig.suptitle(f'{model_name} — Evaluation Plots', fontsize=14)
    plt.tight_layout()
    plt.show()


show_run_plots(RESULTS_DIR / 'yolov8n', 'yolov8n')

## 6. Visualise Predictions on Test Images

In [ ]:
def predict_and_plot(model, img_paths, model_name, conf=0.25, max_imgs=9):
    """Run inference and display predictions with ground-truth boxes side by side."""
    sample = random.sample(img_paths, min(max_imgs, len(img_paths)))
    n_cols = 3
    n_rows = (len(sample) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, sample):
        img_path = Path(img_path)

        # Ground-truth label
        lbl_path = (
            img_path.parent.parent.parent   # yolo/
            / 'labels' / 'test'
            / img_path.with_suffix('.txt').name
        )

        img = np.array(Image.open(img_path).convert('RGB'))
        h, w = img.shape[:2]
        ax.imshow(img)

        # Draw ground-truth (green)
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    _, cx, cy, bw, bh = map(float, line.strip().split())
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    ax.add_patch(patches.Rectangle(
                        (x1, y1), bw * w, bh * h,
                        linewidth=2, edgecolor='lime', facecolor='none',
                        label='GT'
                    ))

        # Draw predictions (red)
        results = model.predict(str(img_path), conf=conf, device=DEVICE, verbose=False)
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf_score = box.conf[0].item()
                ax.add_patch(patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor='red', facecolor='none'
                ))
                ax.text(x1, y1 - 6, f'{conf_score:.2f}',
                        color='red', fontsize=8, fontweight='bold',
                        backgroundcolor='black')

        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    # Hide unused axes
    for ax in axes[len(sample):]:
        ax.axis('off')

    # Legend
    from matplotlib.lines import Line2D
    legend = [
        Line2D([0], [0], color='lime', linewidth=2, label='Ground Truth'),
        Line2D([0], [0], color='red',  linewidth=2, label=f'Prediction (conf>{conf})')
    ]
    fig.legend(handles=legend, loc='lower center', ncol=2, fontsize=11, framealpha=0.9)
    fig.suptitle(f'{model_name} — Predictions on Test Images', fontsize=14, y=1.01)
    plt.tight_layout()

    save_path = RESULTS_DIR / f'{model_name}_test_predictions.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


test_images = sorted(glob.glob(str(TEST_IMG_DIR / '*')))
print(f'Test images found: {len(test_images)}')

random.seed(42)
predict_and_plot(model_nano_best, test_images, 'yolov8n')

## 7. Train YOLOv8s (Small)

YOLOv8s has ~3× more parameters than nano — usually gives a notable mAP boost.

In [ ]:
model_small = YOLO('yolov8s.pt')
print(model_small.info())

In [ ]:
results_small = model_small.train(
    data      = str(DATA_YAML),
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMG_SIZE,
    device    = DEVICE,
    patience  = PATIENCE,
    project   = str(RESULTS_DIR),
    name      = 'yolov8s',
    exist_ok  = True,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.0,
    plots     = True,
    save      = True,
    verbose   = True,
)

print('\nYOLOv8s training complete.')
print(f'Results saved to: {RESULTS_DIR / "yolov8s"}')

## 8. Validate YOLOv8s — Metrics

In [ ]:
best_small = RESULTS_DIR / 'yolov8s' / 'weights' / 'best.pt'
model_small_best = YOLO(str(best_small))

val_small = model_small_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'val',
)

test_small = model_small_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'test',
)

print('\n' + '=' * 50)
print('YOLOv8s — Test Split Metrics')
print('=' * 50)
print(f'  mAP@50      : {test_small.box.map50:.4f}')
print(f'  mAP@50:95   : {test_small.box.map:.4f}')
print(f'  Precision   : {test_small.box.mp:.4f}')
print(f'  Recall      : {test_small.box.mr:.4f}')

In [ ]:
plot_training_curves(RESULTS_DIR / 'yolov8s', 'yolov8s')
show_run_plots(RESULTS_DIR / 'yolov8s', 'yolov8s')

In [ ]:
random.seed(42)
predict_and_plot(model_small_best, test_images, 'yolov8s')

## 9. Compare YOLOv8n vs YOLOv8s

In [ ]:
metrics = {
    'Model':      ['YOLOv8n', 'YOLOv8s'],
    'mAP@50':     [test_nano.box.map50,  test_small.box.map50],
    'mAP@50:95':  [test_nano.box.map,    test_small.box.map],
    'Precision':  [test_nano.box.mp,     test_small.box.mp],
    'Recall':     [test_nano.box.mr,     test_small.box.mr],
    'Weights (MB)': [
        round(os.path.getsize(best_nano) / 1e6, 1),
        round(os.path.getsize(best_small) / 1e6, 1),
    ],
}

df_compare = pd.DataFrame(metrics)
print('=' * 60)
print('YOLOv8 Model Comparison — Test Split')
print('=' * 60)
print(df_compare.to_string(index=False, float_format='{:.4f}'.format))

# Save to CSV for use in evaluation notebook
save_path = RESULTS_DIR / 'yolov8_comparison.csv'
df_compare.to_csv(save_path, index=False)
print(f'\nSaved: {save_path}')

In [ ]:
# Bar chart comparison
metrics_cols = ['mAP@50', 'mAP@50:95', 'Precision', 'Recall']
x  = np.arange(len(metrics_cols))
bw = 0.3

fig, ax = plt.subplots(figsize=(11, 5))
bars_n = ax.bar(x - bw / 2, df_compare.loc[0, metrics_cols], bw,
                label='YOLOv8n', color='steelblue', alpha=0.85)
bars_s = ax.bar(x + bw / 2, df_compare.loc[1, metrics_cols], bw,
                label='YOLOv8s', color='coral',     alpha=0.85)

for bars in [bars_n, bars_s]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('YOLOv8n vs YOLOv8s — Test Split Metrics', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

save_path = RESULTS_DIR / 'yolov8_comparison_chart.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## 10. Save Best Weights to `models/`

In [ ]:
# Copy best.pt and last.pt for both models into models/
def save_model_weights(run_dir, model_name, models_dir):
    run_dir    = Path(run_dir)
    models_dir = Path(models_dir)
    weights    = ['best.pt', 'last.pt']
    for w in weights:
        src = run_dir / 'weights' / w
        if src.exists():
            dst = models_dir / f'{model_name}_{w}'
            shutil.copy2(src, dst)
            size_mb = dst.stat().st_size / 1e6
            print(f'  Saved {dst.name}  ({size_mb:.1f} MB)')
        else:
            print(f'  WARNING: {src} not found')


print('Saving YOLOv8n weights...')
save_model_weights(RESULTS_DIR / 'yolov8n', 'yolov8n', MODELS_DIR)

print('Saving YOLOv8s weights...')
save_model_weights(RESULTS_DIR / 'yolov8s', 'yolov8s', MODELS_DIR)

print('\nAll weights saved to:', MODELS_DIR)
for f in sorted(MODELS_DIR.iterdir()):
    if f.suffix == '.pt':
        print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## 11. Final Summary

In [ ]:
print('=' * 60)
print('YOLOV8 TRAINING SUMMARY')
print('=' * 60)
print(f'Device       : {DEVICE}')
print(f'Epochs       : {EPOCHS} (patience={PATIENCE})')
print(f'Batch size   : {BATCH}')
print(f'Image size   : {IMG_SIZE}')
print()
print(df_compare.to_string(index=False, float_format='{:.4f}'.format))
print()

best_model = df_compare.loc[df_compare['mAP@50'].idxmax(), 'Model']
best_map50 = df_compare['mAP@50'].max()
print(f'Best model   : {best_model}  (mAP@50 = {best_map50:.4f})')
print()
print('Weights saved in:', MODELS_DIR)
print('Plots saved in  :', RESULTS_DIR)
print()
print('=' * 60)
print('NEXT: 04_train_faster_rcnn.ipynb')
print('=' * 60)